# 3 — Asymmetric Cross-Attention Training

Train the **asymmetric** model and compare with the symmetric baseline.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
from training.config import TrainConfig
from training.train import train

## 3.1 — Quick sanity check

In [ ]:
cfg_dev = TrainConfig(
    model_type="asymmetric",
    max_samples=1_000,
    epochs=5,
    batch_size=16,
    run_name="asymmetric_dev",
)
train(cfg_dev)

## 3.2 — Full asymmetric training (3 seeds for statistical significance)

In [ ]:
for seed in [42, 123, 7]:
    cfg = TrainConfig(
        model_type="asymmetric",
        max_samples=None,
        epochs=20,
        batch_size=64,
        seed=seed,
        run_name=f"asymmetric_s{seed}",
    )
    train(cfg)

## 3.3 — Also run symmetric with multiple seeds

In [ ]:
for seed in [42, 123, 7]:
    cfg = TrainConfig(
        model_type="symmetric",
        max_samples=None,
        epochs=20,
        batch_size=64,
        seed=seed,
        run_name=f"symmetric_s{seed}",
    )
    train(cfg)

## 3.4 — Compare best results

In [ ]:
from training.evaluate import evaluate_from_checkpoint
from visualization.plot_results import print_comparison_table
import numpy as np

results = {}
for model_type in ["asymmetric", "symmetric"]:
    accs = []
    for seed in [42, 123, 7]:
        ckpt = f"../results/checkpoints/{model_type}_s{seed}_best.pt"
        m = evaluate_from_checkpoint(ckpt, data_dir="../data")
        accs.append(m)
    # Average over seeds
    avg = {k: np.mean([a[k] for a in accs]) for k in accs[0]}
    std = {k: np.std([a[k] for a in accs]) for k in accs[0]}
    results[model_type.title()] = avg
    print(f"\n{model_type.title()}:")
    for k in avg:
        print(f"  {k}: {avg[k]:.2f} +/- {std[k]:.2f}")

print("\n")
print_comparison_table(results)